# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`
- **tara_n1_pretrain.pth:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth`
## Completed training phases:
- ✅Completed first 100M - resulting 100M token corpus
- ✅Completed next 100M - resulting 200M token corpus
- ✅Completed next 100M - resulting 300M token corpus
- ✅Completed next 100M - resulting 400M token corpus
- ✅Completed next 100M - resulting 500M token corpus
- ✅Completed next 500M - resulting 1B token corpus
- ✅Completed next 500M - resulting 1.5B token corpus
- ✅Completed next 500M - resulting 2B token corpus
- ✅Completed next 500M - resulting 2.5B token corpus
- ✅Completed next 500M - resulting 3B token corpus
- ✅Completed next 1B - resulting 4B token corpus
- Completed next 1B - resulting 5B token corpus

In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
with open("tara_n1_pretrain.pth", "wb") as f:
    f.write(model_res.content)
print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded train_utils.py successfully.
Downloaded wieghts


In [3]:
from model import *
from train_utils import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=256,
    batch_size=64,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

In [5]:
# modelV1 = CustomGPT(config)

# if torch.cuda.device_count() > 1:
#     modelV1 = nn.DataParallel(modelV1)
#     print(f"Using {torch.cuda.device_count()} GPUs")

# modelV1.to(device)

# calc_params(modelV1)

# loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
# scaler = torch.amp.GradScaler()


modelV2 = CustomGPT(config)
# state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
# state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

modelV2.load_weights("tara_n1_pretrain.pth")
# modelV2.load_weights("Models/tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=2e-4, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler()

Loaded weights from tara_n1_pretrain.pth
Using 2 GPUs
Total Parameters: 30,586,449
Trainable Parameters: 30,586,449


# The Dataset

In [6]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
# target_tokens = 100_000_000
# skip_tokens = 0

# skip_tokens = 100_000_000
# target_tokens = 200_000_000
# fname = "tokens_100m_to_200m.bin" 

# skip_tokens = 200_000_000
# target_tokens = 300_000_000
# fname = "tokens_200m_to_300m.bin"

skip_tokens = 4_000_000_000
train_ds = load_dataset("HuggingFaceFW/fineweb", split="train", name="sample-10BT", streaming=True)
test_ds = load_dataset("HuggingFaceFW/fineweb", split="train", name="sample-10BT", streaming=True)


train_dataset = StreamingDataset(train_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64, skip_tokens = skip_tokens)
test_dataset = StreamingDataset(test_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64)


train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 0)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 0)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

# Pretraining the model

In [7]:
from tqdm.auto import tqdm
steps = 20000
testing_step = 1000
train_iter = iter(train_dataloader)
avg_train_loss = 0
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device)
    avg_train_loss += train_loss
    
    if step % testing_step == 0:
        # modelV1.eval()
        avg_train_loss /= testing_step
        modelV2.eval()
        
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {avg_train_loss} | Test Loss: {test_loss:}")
        avg_train_loss = 0

  0%|          | 0/20000 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Step 1000 | Train Loss: 4.381619453430176 | Test Loss: 4.439255285263061


  0%|          | 0/20 [00:00<?, ?it/s]

Step 2000 | Train Loss: 4.45768928527832 | Test Loss: 4.495054244995117


...

Step 18000 | Train Loss: 4.612350940704346 | Test Loss: 4.630600142478943


  0%|          | 0/20 [00:00<?, ?it/s]

Step 19000 | Train Loss: 4.577840328216553 | Test Loss: 4.622191452980042


  0%|          | 0/20 [00:00<?, ?it/s]

Step 20000 | Train Loss: 4.594461917877197 | Test Loss: 4.620356202125549


In [8]:
# torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v3.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v4.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v5.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v6.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v7.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v8.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v9.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v10.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v11.pth")
torch.save(modelV2.state_dict(), "tara_n1_pretrain_v12.pth")

# Testing



In [9]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v12.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")


Loaded weights from tara_n1_pretrain_v12.pth
Input:
Once upon a time, 

Output:
Once upon a time, ].
Can bin be avoided when aromatherapy cences ingredientados exists?
Great for the condition of the plant or Server, and for the findings into its only use, when collection Nineterm preservation efforts arise. But it may be vital that the soil in each plant are planted attacks on small Provide humane grass grown drivers for outside use. The then Commercial produce is evidence of the eye for movement approaches due to the influx there. The plant’s calibrating the Toters and Feeding System and Attention For NEVER wood cavity, an Eitherectine is used. There is no batch of leafes available, but a second Tbsp of if a pen and rinsed leaves of feet, and a good effect on the seed. For these purposes, your plants will lose their buds if they fail in season 4-11 and. (This is a problem because of unsatisfactory, antique and requires the approval of a request!)
|Please note that we are also closed f